# 🏛️ oracc-parser — Quickstart

This notebook shows you the actual data available, how to parse it, and where everything lives on disk.

## 🎯 Goals of this notebook

1. **Confirm data is available** — check that downloaded project ZIPs are present
2. **Understand the directory layout** — data, cache, output, jsonzip directories
3. **Browse available projects** — see what's downloaded locally vs. what exists on ORACC
4. **Parse a project** — run the full pipeline on a real ORACC corpus
5. **Explore tablets** — inspect transliterations, metadata, and sign-level Unicode cuneiform
6. **See the reference data** — provenance, periods, sign list

## 1. Get the data (Zenodo)

First, let's make sure you have the pre-downloaded data (project ZIPs, translations, Pleiades).
This downloads ~770MB from Zenodo so you don't have to scrape ORACC yourself.

In [1]:
import os
from pathlib import Path
from oracc_parser.settings import jsonzip_dir

# Check if data exists
zip_dir = jsonzip_dir()
if not zip_dir.exists() or len(list(zip_dir.glob("*.zip"))) == 0:
    print("⬇️  Data missing. Downloading from Zenodo... (this may take a minute)")
    # Run the download script (works in Jupyter)
    %run ../scripts/download_zenodo_data.py
else:
    print(f"✅ Data found in {zip_dir} ({len(list(zip_dir.glob('*.zip')))} projects)")

✅ Data found in C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\jsonzip (146 projects)


## 2. What data do we have?

Let's look at what's already downloaded and what's available.

In [4]:
from pathlib import Path
from oracc_parser import reference_data
from oracc_parser.settings import data_dir, output_dir, cache_dir, jsonzip_dir

# What directories does oracc-parser use?
print("📁 Directory layout:")
print(f"   Data dir:    {data_dir()}")
print(f"   JSON ZIPs:   {jsonzip_dir()}")
print(f"   Output dir:  {output_dir()}")
print(f"   Cache dir:   {cache_dir()}")

📁 Directory layout:
   Data dir:    C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data
   JSON ZIPs:   C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\jsonzip
   Output dir:  C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\output
   Cache dir:   C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\enriched_data\cache


In [5]:
# What ORACC projects exist in the world?
# We fetch the LIVE list from ORACC servers:
all_projects = reference_data.get_live_project_list()
print(f"🌍 ORACC has {len(all_projects)} public projects")
all_projects.head(10)

19:20:36 | INFO    | oracc_parser | Fetching live project list (attempt 1/3)...
C:\Users\shaha\PycharmProjects\thesis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'oracc.museum.upenn.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
19:20:38 | WARNING | oracc_parser | Detected malformed ORACC JSON, attempting regex fix...
19:20:38 | INFO    | oracc_parser | Fetched 144 projects from ORACC live list


🌍 ORACC has 144 public projects


,project,name,abbrev,blurb
0,edlex,Early Dynastic Lexical Texts,EDLEX,Editions and translations of Early Dynastic le...
1,ribo/babylon4,The Inscriptions of the Bazi Dynasty,Babylon 4,This sub-project includes editions of the offi...
2,ribo/babylon10,The Borsippa Inscription of Antiochus I Soter,Babylon 10,This sub-project includes an edition of the Bo...
3,ribo/babylon5,The Inscriptions of the Elamite Dynasty,Babylon 5,This sub-project includes editions of the offi...
4,ribo/babylon6,The Inscriptions of the Period of the Uncertai...,Babylon 6,This sub-project includes editions of the offi...
5,ribo/bab7scores,Scores of the Inscriptions of the Neo-Babyloni...,Scores,This sub-project presently includes score tran...
6,ribo,Royal Inscriptions of Babylonia online,RIBo,This project intends to present annotated edit...
7,ribo/babylon8,The Inscriptions of Cyrus II and His Successors,Babylon 8,This sub-project presently includes editions o...
8,ribo/babylon3,The Inscriptions of the Second Dynasty of the ...,Babylon 3,This sub-project includes editions of the offi...
9,ribo/babylon7,The Inscriptions of the Neo-Babylonian Dynasty,Babylon 7,This sub-project presently includes editions o...


In [6]:
# Which ones have we already downloaded?
import os
from oracc_parser.settings import jsonzip_dir

# Check for downloaded ZIPs using configured path
zip_dir = jsonzip_dir()

if zip_dir.exists():
    downloaded = sorted([f.stem for f in zip_dir.glob("*.zip")])
    total_size_mb = sum(f.stat().st_size for f in zip_dir.glob("*.zip")) / (1024*1024)
    print(f"📦 {len(downloaded)} projects downloaded ({total_size_mb:.0f} MB total)")
    print()
    # Show first 20
    for i, name in enumerate(downloaded[:20]):
        size_mb = (zip_dir / f"{name}.zip").stat().st_size / (1024*1024)
        print(f"   {name:40s}  {size_mb:6.1f} MB")
    if len(downloaded) > 20:
        print(f"   ... and {len(downloaded)-20} more")
else:
    print("No downloaded projects found yet.")
    print("Run: oracc-parser download --project saao/saa01")

📦 146 projects downloaded (839 MB total)

   adsd                                         8.7 MB
   adsd-adart1                                  4.8 MB
   adsd-adart2                                  7.0 MB
   adsd-adart3                                 10.2 MB
   adsd-adart5                                  3.8 MB
   adsd-adart6                                  4.6 MB
   aemw                                         0.0 MB
   aemw-alalakh-idrimi                          0.2 MB
   aemw-amarna                                  7.7 MB
   aemw-ugarit                                 16.5 MB
   akklove                                      2.1 MB
   ario                                         2.6 MB
   asbp-ninmed                                  9.8 MB
   asbp-rlasb                                   0.2 MB
   atae                                        76.2 MB
   atae-assurmisc                               0.2 MB
   atae-burmarina                               0.3 MB
   atae-ctn1           

In [7]:
# Compare: which projects are NOT yet downloaded?
if zip_dir.exists() and 'all_projects' in dir():
    downloaded_names = {f.stem.replace('-', '/') for f in zip_dir.glob('*.zip')}
    live_names = set(all_projects['project'].tolist()) if 'project' in all_projects.columns else set()
    
    not_downloaded = sorted(live_names - downloaded_names)
    
    print(f"📊 Download status:")
    print(f"   ✅ {len(downloaded_names)} projects downloaded")
    print(f"   ❌ {len(not_downloaded)} projects NOT yet downloaded")
    print(f"   🌍 {len(live_names)} total projects on ORACC")
    print()
    if not_downloaded:
        print("Projects not yet downloaded:")
        for name in not_downloaded:
            print(f"   • {name}")
        print()
        print("To download one: oracc-parser download --project <name>")
    else:
        print("🎉 All available projects are downloaded!")


📊 Download status:
   ✅ 146 projects downloaded
   ❌ 48 projects NOT yet downloaded
   🌍 144 total projects on ORACC

Projects not yet downloaded:
   • amgg
   • armep
   • arrim
   • asbp
   • atae/assur
   • atae/durszarrukin
   • atae/kalhu
   • atae/kunalia
   • atae/nineveh
   • balt
   • btmao
   • cams/akno
   • cmawro
   • contrib/jacobsen
   • contrib/lambert
   • ctij
   • dcclt
   • dcclt/ebla
   • dcclt/jena
   • dcclt/nineveh
   • dcclt/signlists
   • dccmt
   • doc
   • dsst
   • ecut
   • edlex
   • eisl
   • epsd2
   • etcsri
   • iraq
   • iraq/iraq85
   • kish
   • kish/fieldmus
   • kish/fieldmus/fmod
   • kish/mathaffield
   • neo
   • obel
   • obmc
   • ogsl
   • oimea
   • osl
   • pnao
   • qcat
   • riao
   • saao/aebp
   • saao/knpp
   • tsae
   • xcat

To download one: oracc-parser download --project <name>


In [8]:
# ⬇️ Download Data
# You can download any project that was not yet downloaded by us, by name.
from oracc_parser.download.oracc_download import download_projects

projects_to_download = [
   "ctij"
]


if projects_to_download:
    download_projects(projects_to_download)


19:20:48 | INFO    | oracc_parser | Downloading 1 project(s)...
19:20:49 | ERROR   | oracc_parser | Failed to download ctij: 500 Server Error: Internal Server Error for url: https://oracc.museum.upenn.edu//json/ctij.zip
19:20:49 | INFO    | oracc_parser | Downloaded 0/1 projects.


## 3. Parse a project

Pick any project from the lists above. We'll parse a small sample to see the output.

In [7]:
from oracc_parser import parse_project, RunConfig
from oracc_parser.settings import jsonzip_dir

# Change this to any project you want!
PROJECT = "saao/saa01"  
LIMIT = 5  # Set to None to parse everything

# Check that the project is available locally
zip_dir = jsonzip_dir()
zip_name = PROJECT.replace('/', '-') + '.zip'
if not (zip_dir / zip_name).exists():
    available = sorted(f.stem.replace('-', '/') for f in zip_dir.glob('*.zip'))
    print(f"⚠️  Project '{PROJECT}' is not downloaded!")
    print(f"   Available projects ({len(available)} total): {available[:10]}...")
    print(f"   To download: oracc-parser download --project {PROJECT}")
else:
    config = RunConfig(limit=LIMIT)
    records = parse_project(PROJECT, config=config)
    print(f"✓ Parsed {len(records)} tablets from {PROJECT}")

19:16:02 | INFO    | oracc_parser | Already downloaded: c:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\data\jsonzip\saao-saa01.zip
Parsing saao/saa01: 100%|██████████| 5/5 [00:00<00:00,  5.56it/s]
19:16:04 | INFO    | oracc_parser | Parsed 5 tablets from saao/saa01


✓ Parsed 5 tablets from saao/saa01


> **💡 Caching:** The first `parse_project()` call parses everything (slow).
> Subsequent calls with the **same config** return instantly from cache.
> Switching configs (e.g. `mask_pos`, `drop_missing`) reuses the cached words
> and only rebuilds the string representations — still much faster than re-parsing.
> See **Notebook 03** for details, or use `RunConfig(use_cache=False)` to force re-parsing.

In [8]:
# Look at one tablet up close
tablet = records[0]

print(f"=== {tablet.metadata.identifier} ===")
print(f"   Project:     {PROJECT}")
print(f"   Text ID:     {tablet.metadata.id_text}")
print(f"   Genre:       {tablet.metadata.genre}")

geo = tablet.metadata.geographical_information
if geo:
    print(f"   Provenance:  {geo.city.city_name}")
    print(f"   Pleiades ID: {geo.city.city_plaides_id}")

chrono = tablet.metadata.chronological_information
if chrono:
    print(f"   Period:      {chrono.tablet_period.period_name if chrono.tablet_period else ''}")
    print(f"   Years:       {chrono.start_year} to {chrono.end_year}")

print(f"   Words:       {len(tablet.content.words)}")

=== saao/saa01_P314227 ===
   Project:     saao/saa01
   Text ID:     P314227
   Genre:       Administrative Letter
   Provenance:  Nineveh
   Pleiades ID: 874621
   Period:      Neo-Assyrian
   Years:       -911 to -612
   Words:       108


In [9]:
# The text in different representations
c = tablet.content

print("TRANSLITERATION:")
print(f"{c.transliterated_str_representation.text[:200] if c.transliterated_str_representation else '(none)'}")
print()
print("NORMALIZATION:")
print(f"{c.normalized_str_representation.text[:200] if c.normalized_str_representation else '(none)'}")
print()
print("UNICODE CUNEIFORM:")
print(f"{c.unicode_str_representation.text[:200] if c.unicode_str_representation else '(none)'}")
print()
print("ENGLISH TRANSLATION:")
print(f"{c.english_translation[:200] if c.english_translation else '(none)'}")

TRANSLITERATION:
[a-na LUGAL be-li₂]-ia₂
[ARAD-ka {1}ki-ṣir—aš-šur lu] DI-mu a-⸢na⸣
[LUGAL be-li₂-ia₂ ša LUGAL] ⸢be⸣-li₂ iš-⸢pur-an-ni⸣
[ma-a x x x x]-ia ina IGI
[x x x x x x]+⸢x⸣ a-na-ku
[x x x x up]-⸢ta⸣-at-ti\t-iu-

NORMALIZATION:
ana šarri bēlīya
urdaka Kiṣir-Aššur lū šulmu ana
šarri bēlīya ša šarru bēlī išpuranni
mā UNK UNK UNK UNK ina pān
UNK UNK UNK UNK UNK UNK anāku
UNK UNK UNK UNK uptattiʾu
UNK UNK UNK UNK šapluššu iptaqd

UNICODE CUNEIFORM:
𒀀𒈾 𒈗 𒁁𒉌𒐊
𒀴𒅗 𒁹𒆠𒈲𒀸𒋩 𒇻 𒁲𒈬 𒀀𒈾
𒈗 𒁁𒉌𒐊 𒊭 𒈗 𒁁𒉌 𒅖𒁓𒀭𒉌
𒈠𒀀 x x x x𒅀 𒀸 𒅆
x x x x x x 𒀀𒈾𒆪
x x x x 𒌒𒋫𒀜𒋾𒅀𒌋
x x x x 𒆠𒋫𒍑𒋙 𒅁𒋳𒁺
x x x x𒉌 𒌑𒊓𒄭𒊒
x x x 𒌒𒋫𒀜𒋾𒅀𒌋
x x x 𒁳𒋢𒉺𒎌 𒁳𒋢𒉺𒎌 𒋗𒌑
x x x x x 𒌷𒀏 x x
x x x x𒎌 𒊭 𒌷𒊮𒌷 x x𒇷𒋙𒉡
x x x

ENGLISH TRANSLATION:
[To the king], my [lord: your servant Kiṣir-Aššur. Good] health to [the king, my lord!
As to what the king] my lord wrote to me:
"[......] in the presence
[of......]!" I
[......] they discharged
[....


## 5. Get flat DataFrames (for analysis & export)

Each function returns a clean pandas DataFrame — no nesting, no Pydantic.

In [10]:
from oracc_parser import get_metadata_table, get_transliterations, get_full_flat_table

# Metadata table
print("=== METADATA ===")
meta = get_metadata_table(records)
display(meta)

=== METADATA ===


,id,project,text_id,genre,provenance,pleiades_id,state_supergroup,period,start_year,end_year
0,saao/saa01_P314227,saao/saa01,P314227,Administrative Letter,Nineveh,874621,,Neo-Assyrian,-911,-612
1,saao/saa01_P313458,saao/saa01,P313458,Administrative Letter,Nineveh,874621,,Neo-Assyrian,-911,-612
2,saao/saa01_P334317,saao/saa01,P334317,Administrative Letter,Nineveh,874621,,Neo-Assyrian,-911,-612
3,saao/saa01_P314232,saao/saa01,P314232,Administrative Letter,Nineveh,874621,,Neo-Assyrian,-911,-612
4,saao/saa01_P224487,saao/saa01,P224487,Administrative Letter,Nimrud,894019,,Neo-Assyrian,-911,-612


In [11]:
# Transliterations
print("=== TRANSLITERATIONS ===")
trans = get_transliterations(records)
display(trans)

=== TRANSLITERATIONS ===


,id,project,transliteration,total_tokens,tokens_without_broken
0,saao/saa01_P314227,saao/saa01,[a-na LUGAL be-li₂]-ia₂\n[ARAD-ka {1}ki-ṣir—aš...,107,62
1,saao/saa01_P313458,saao/saa01,[a-na LUGAL] EN-⸢ia⸣ ARAD-⸢ka⸣ {1}hu-un-ni-i]\...,179,167
2,saao/saa01_P334317,saao/saa01,13 KUŠ₃ mu-lu-u 04 KUŠ₃ DAGAL E₂—dan-ni\n[x KU...,67,67
3,saao/saa01_P314232,saao/saa01,[a]-⸢na⸣ EN.⸢NUN-ku-nu la ta⸣-[ši-ṭa EN.NUN-ku...,297,161
4,saao/saa01_P224487,saao/saa01,[a-na LUGAL] EN-⸢ia⸣\n[ARAD-ka {1}{⸢d}MES⸣—rem...,206,170


In [12]:
# Full flat table — everything in one DataFrame
flat = get_full_flat_table(records)
print(f"Full table: {flat.shape[0]} rows × {flat.shape[1]} columns")
print(f"Columns: {list(flat.columns)}")
display(flat)

Full table: 5 rows × 15 columns
Columns: ['id', 'project', 'text_id', 'genre', 'provenance', 'period', 'start_year', 'end_year', 'transliteration', 'normalization', 'lemmatization', 'unicode', 'translation', 'total_tokens', 'tokens_without_broken']


,id,project,text_id,genre,provenance,period,start_year,end_year,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,saao/saa01_P314227,saao/saa01,P314227,Administrative Letter,Nineveh,Neo-Assyrian,-911,-612,[a-na LUGAL be-li₂]-ia₂\n[ARAD-ka {1}ki-ṣir—aš...,ana šarri bēlīya\nurdaka Kiṣir-Aššur lū šulmu ...,ana šarru bēlu\nardu Kiṣir-Aššur lū šulmu ana\...,𒀀𒈾 𒈗 𒁁𒉌𒐊\n𒀴𒅗 𒁹𒆠𒈲𒀸𒋩 𒇻 𒁲𒈬 𒀀𒈾\n𒈗 𒁁𒉌𒐊 𒊭 𒈗 𒁁𒉌 𒅖𒁓𒀭𒉌\...,"[To the king], my [lord: your servant Kiṣir-Aš...",107,62
1,saao/saa01_P313458,saao/saa01,P313458,Administrative Letter,Nineveh,Neo-Assyrian,-911,-612,[a-na LUGAL] EN-⸢ia⸣ ARAD-⸢ka⸣ {1}hu-un-ni-i]\...,ana šarri bēlīya urdaka Hunni\nlū šulmu ana ša...,ana šarru bēlu ardu Hunni\nlū šulmu ana šarru ...,𒀀𒈾 𒈗 𒂗𒅀 𒀴𒅗 𒁹𒄷𒌦𒉌𒄿\n𒇻𒌋 𒂄𒈬 𒀀𒈾 𒈗 𒂗𒐊 𒀭𒀝 𒌋 𒀭𒀫𒌓 𒀀𒈾 𒈗 ...,[To the king m]y lord: your servant [Hunnî]. G...,179,167
2,saao/saa01_P334317,saao/saa01,P334317,Administrative Letter,Nineveh,Neo-Assyrian,-911,-612,13 KUŠ₃ mu-lu-u 04 KUŠ₃ DAGAL E₂—dan-ni\n[x KU...,UNK ammāti mūlû UNK ammāti rupšu bēt danni\nUN...,UNK ammatu mūlû UNK ammatu rupšu bītu dannu\nU...,𒌋𒐈 𒌑 𒈬𒇻𒌋 𒐉 𒌑 𒂼 𒂍𒆗𒉌 𒂍𒆗𒉌\nx 𒌑 𒈬𒇻𒌋 𒐉 𒌑 𒂼 𒂍 𒍇𒇻\nx ...,"Height[ 1]3 cubits (6,5 m), width four cubits ...",67,67
3,saao/saa01_P314232,saao/saa01,P314232,Administrative Letter,Nineveh,Neo-Assyrian,-911,-612,[a]-⸢na⸣ EN.⸢NUN-ku-nu la ta⸣-[ši-ṭa EN.NUN-ku...,ana maṣṣartikunu lā tašīṭa maṣṣartakunu\nlū da...,ana maṣṣartu lā šâṭu maṣṣartu\nlū dannu laššu ...,𒀀𒈾 𒂗𒉣𒆪𒉡 𒆷 𒋫𒅆𒁕 𒂗𒉣𒆪𒉡\n𒇻 𒆗𒈾𒀜 𒆷𒀾𒋗 𒈠x x x x\n𒉌𒆤𒋫𒆪𒉡 ...,Do not neglect your guard; [you] must be inten...,297,161
4,saao/saa01_P224487,saao/saa01,P224487,Administrative Letter,Nimrud,Neo-Assyrian,-911,-612,[a-na LUGAL] EN-⸢ia⸣\n[ARAD-ka {1}{⸢d}MES⸣—rem...,ana šarri bēlīya\nurdaka Marduk-remanni\nlū šu...,ana šarru bēlu\nardu Marduk-remanni\nlū šulmu ...,𒀀𒈾 𒈗 𒂗𒅀\n𒀴𒅗 𒁹𒀭𒈩𒀖𒀀𒉌\n𒇻 𒁲𒈬 𒀀𒈾 𒈗 𒂗𒅀\n𒄿𒌍𒉡 𒂊𒉿𒅖\n𒀭x ...,"[To the king], my lord: [your servant M]arduk-...",206,170


## 6. Export & see where output goes

In [13]:
# Export as JSONL
out_dir = output_dir()
out_dir.mkdir(parents=True, exist_ok=True)

jsonl_path = out_dir / f"{PROJECT.replace('/', '_')}.jsonl"
csv_path = out_dir / f"{PROJECT.replace('/', '_')}.csv"

flat.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
flat.to_csv(csv_path, index=False)

print(f"✓ Exported to:")
print(f"   JSONL: {jsonl_path}  ({jsonl_path.stat().st_size / 1024:.1f} KB)")
print(f"   CSV:   {csv_path}  ({csv_path.stat().st_size / 1024:.1f} KB)")
print()
print(f"📁 Output directory: {out_dir}")
for f in sorted(out_dir.iterdir()):
    print(f"   {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

✓ Exported to:
   JSONL: C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\output\saao_saa01.jsonl  (28.1 KB)
   CSV:   C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\output\saao_saa01.csv  (26.7 KB)

📁 Output directory: C:\Users\shaha\PycharmProjects\thesis\oracc_preprocessing\oracc-parser\output
   saao_saa01.csv                            26.7 KB
   saao_saa01.jsonl                          28.1 KB


## 7. Reference data (bundled with the package)

These datasets ship with oracc-parser — no download needed.

In [14]:
from oracc_parser import reference_data

print("Bundled reference data:")
datasets = {
    "Provenance (cities → Pleiades IDs)": reference_data.get_provenance(),
    "Period mapping (period → years)": reference_data.get_period_mapping(),
    "Sign readings (8900+ signs)": reference_data.get_sign_list(),
    "POS tags": reference_data.get_pos_tags(),
    "Languages": reference_data.get_languages(),
    "ORACC projects metadata": reference_data.get_projects_metadata(),
}
for name, df in datasets.items():
    print(f"   {name:45s}  {len(df):>6} rows × {len(df.columns)} cols")

Bundled reference data:
   Provenance (cities → Pleiades IDs)                333 rows × 6 cols
   Period mapping (period → years)                    21 rows × 3 cols
   Sign readings (8900+ signs)                      8901 rows × 2 cols
   POS tags                                           56 rows × 5 cols
   Languages                                          49 rows × 7 cols
   ORACC projects metadata                           221 rows × 7 cols


In [15]:
# Explore any of them
print("=== Provenance ===")
display(reference_data.get_provenance().head(10))

print("\n=== Period mapping ===")
display(reference_data.get_period_mapping())

=== Provenance ===


,raw_provenience,normalized_city,pleiades_id,pleiades_title,lat,lon
0,***,Uncertain,-,NaN,NaN,NaN
1,Acharneh (Tunip),Tunip,668200,Asharne,35.2833333,36.4
2,Adab,Adab,787747618,NaN,NaN,NaN
3,Akhetaten (Mod. Amarna),el-Amarna,149576487,Amarna,27.644099597341675,30.89818203754354
4,Akhetaten (mod. el-Amarna),el-Amarna,149576487,Amarna,27.644099597341675,30.89818203754354
5,Akko,Akko,678010,Ake/Ptolemais,32.92128675,35.079835
6,Akšapa,Akšapa,?,NaN,NaN,NaN
7,Alalakh,Alalakh,309866128,Alalaḫ,36.23807544872566,36.38383192027712
8,Alashiya,NaN,NaN,NaN,NaN,NaN
9,Amarna,el-Amarna,149576487,Amarna,27.644099597341675,30.89818203754354



=== Period mapping ===


,period_name,start_year,end_year
0,hellenistic,-323,-30
1,achaemenid,-550,-330
2,Neo-Babylonian,-626,-539
3,Neo-Assyrian,-911,-612
4,Parthian,-247,224
5,Middle Babylonian,-1595,-1155
6,Middle Assyrian,-1363,-912
7,Old Babylonian,-1894,-1595
8,Old Akkadian,-2334,-2154
9,Early Old Babylonian,-1894,-1595


## What's next?

- **Clean the data:** See notebook `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **Explore reference data:** See notebook `02_reference_data.ipynb` for a catalog of available projects, pos tags, and languages.